# Lecture 3: Loss Functions


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, bernoulli
from scipy.special import expit  # sigmoid

np.random.seed(42)
print("Libraries loaded.")


## 1. Maximum Likelihood Estimation (Ch. 5.1)

The core idea of Ch. 5.1:

> Choose model parameters `θ` to **maximize the probability** of observing the data.

$$\\hat{\\theta} = \\arg\\max_\\theta \\prod_{i=1}^N p(y_i | \\mathbf{x}_i, \\theta)$$

Taking the log converts the product to a sum:

$$\\hat{\\theta} = \\arg\\max_\\theta \\sum_{i=1}^N \\log p(y_i | \\mathbf{x}_i, \\theta)$$

**Minimizing the loss = minimizing the negative log-likelihood**


In [ ]:
# 1D Gaussian likelihood visualization
# Observation: y=2.5, sigma=1 fixed
# Compute likelihood by scanning parameter mu

y_obs = 2.5
sigma = 1.0
mu_range = np.linspace(-1, 6, 300)

likelihood = norm.pdf(y_obs, loc=mu_range, scale=sigma)
log_likelihood = norm.logpdf(y_obs, loc=mu_range, scale=sigma)
nll = -log_likelihood  # Negative log-likelihood (loss)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(mu_range, likelihood, 'steelblue', lw=2)
axes[0].axvline(y_obs, color='red', linestyle='--', label=f"y_obs={y_obs}")
axes[0].set_title("Likelihood  p(y|mu)")
axes[0].set_xlabel("mu (model parameter)"); axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(mu_range, log_likelihood, 'darkorange', lw=2)
axes[1].axvline(y_obs, color='red', linestyle='--', label=f"maximum: mu={y_obs}")
axes[1].set_title("Log-Likelihood  log p(y|mu)")
axes[1].set_xlabel("mu"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(mu_range, nll, 'crimson', lw=2)
axes[2].axvline(y_obs, color='red', linestyle='--', label=f"minimum: mu={y_obs}")
axes[2].set_title("Negative Log-Likelihood (LOSS)")
axes[2].set_xlabel("mu"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle("MLE: Likelihood -> Log-Likelihood -> Loss", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb03_mle.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Regression Loss: MSE

Under the Gaussian noise assumption, NLL reduces to **Mean Squared Error (MSE)**:

$$L = \\frac{1}{N} \\sum_{i=1}^N (y_i - f_\\theta(x_i))^2$$


In [ ]:
# Compare different loss functions: MSE vs MAE vs Huber
residuals = np.linspace(-3, 3, 300)

mse   = residuals**2
mae   = np.abs(residuals)
huber = np.where(np.abs(residuals) < 1,
                 0.5 * residuals**2,
                 np.abs(residuals) - 0.5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(residuals, mse,   'steelblue',  lw=2, label="MSE (L2)")
axes[0].plot(residuals, mae,   'darkorange', lw=2, label="MAE (L1)")
axes[0].plot(residuals, huber, 'crimson',    lw=2, label="Huber")
axes[0].set_title("Regression Loss Functions")
axes[0].set_xlabel("Residual (y - y_hat)"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

# MSE vs outliers
np.random.seed(5)
x_data = np.linspace(0, 10, 50)
y_clean = 2*x_data + 1 + np.random.randn(50)*1.5
y_outlier = y_clean.copy()
y_outlier[[10, 25, 40]] += np.array([15, -12, 18])  # Outliers

coef_clean   = np.polyfit(x_data, y_clean, 1)
coef_outlier = np.polyfit(x_data, y_outlier, 1)

axes[1].scatter(x_data, y_outlier, alpha=0.6, color='gray', label="Data (with outliers)")
axes[1].scatter(x_data[[10,25,40]], y_outlier[[10,25,40]],
                color='red', s=80, zorder=5, label="Outlier")
axes[1].plot(x_data, np.polyval(coef_clean,   x_data), 'green',  lw=2, label="Clean data regression")
axes[1].plot(x_data, np.polyval(coef_outlier, x_data), 'crimson', lw=2, linestyle='--',
             label="Regression with outliers")
axes[1].set_title("MSE is Sensitive to Outliers!")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb03_regression_loss.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Binary Classification: Binary Cross-Entropy

Under the Bernoulli distribution assumption, NLL reduces to **Binary Cross-Entropy**:

$$L = -\\frac{1}{N}\\sum_{i=1}^N \\left[ y_i \\log(\\hat{p}_i) + (1-y_i) \\log(1-\\hat{p}_i) \\right]$$

The model output is mapped to [0,1] via the sigmoid function:

$$\\hat{p} = \\sigma(f_\\theta(x)) = \\frac{1}{1+e^{-f_\\theta(x)}}$$


In [ ]:
# Binary Cross-Entropy visualization
p_hat = np.linspace(0.001, 0.999, 300)

bce_y1 = -np.log(p_hat)       # loss when y=1
bce_y0 = -np.log(1 - p_hat)   # loss when y=0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(p_hat, bce_y1, 'steelblue', lw=2, label="y=1: -log(p_hat)")
axes[0].plot(p_hat, bce_y0, 'crimson',   lw=2, label="y=0: -log(1-p_hat)")
axes[0].set_title("Binary Cross-Entropy Components")
axes[0].set_xlabel("Predicted probability p_hat")
axes[0].set_ylabel("Loss"); axes[0].set_ylim(0, 5)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Simulated sigmoid classifier
np.random.seed(1)
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X_cls, y_cls = make_classification(n_samples=200, n_features=2,
                                    n_redundant=0, n_clusters_per_class=1, random_state=42)
clf = LogisticRegression()
clf.fit(X_cls, y_cls)

xx, yy = np.meshgrid(np.linspace(X_cls[:,0].min()-0.5, X_cls[:,0].max()+0.5, 200),
                     np.linspace(X_cls[:,1].min()-0.5, X_cls[:,1].max()+0.5, 200))
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.7)
axes[1].scatter(X_cls[:,0], X_cls[:,1], c=y_cls, cmap='RdBu_r', edgecolors='k', s=30)
axes[1].set_title("Logistic Classifier with BCE\n(decision boundary + probability map)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb03_bce.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Multi-Class Classification: Softmax + Cross-Entropy

For K classes:

$$\\text{Softmax}(z_k) = \\frac{e^{z_k}}{\\sum_{j=1}^K e^{z_j}}$$

$$L = -\\frac{1}{N}\\sum_{i=1}^N \\log \\hat{p}_{y_i,i}$$


In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))  # numerical stability
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(y_true_idx, probs):
    N = len(y_true_idx)
    return -np.mean(np.log(probs[np.arange(N), y_true_idx] + 1e-9))

# Example: 3 classes, 5 samples
np.random.seed(3)
logits = np.random.randn(5, 3) * 2
probs = softmax(logits)
y_true = np.array([0, 2, 1, 0, 2])

print("Logits (raw model output):")
print(logits.round(2))
print()
print("Softmax probabilities (row sums = 1):")
print(probs.round(3))
print(f"\nRow sums: {probs.sum(axis=1).round(4)}")
print(f"\nCross-Entropy Loss: {cross_entropy(y_true, probs):.4f}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i in range(3):
    colors = ['crimson' if j == y_true[i] else 'steelblue' for j in range(3)]
    axes[i % 3].bar(['Class 0','Class 1','Class 2'], probs[i], color=colors, edgecolor='white')
    axes[i % 3].set_title(f"Sample {i}: True class={y_true[i]}")
    axes[i % 3].set_ylabel("Probability")
    axes[i % 3].set_ylim(0, 1)
    axes[i % 3].grid(axis='y', alpha=0.3)
    axes[i % 3].set_xlabel("Red = correct class")

plt.suptitle("Softmax Output Probabilities (3 samples)", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb03_softmax.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Loss Function Summary Table

| Problem | Distribution | Loss Function |
|---|---|---|
| Regression | Gaussian | MSE (L2) |
| Regression (robust) | Laplace | MAE (L1) |
| Binary classification | Bernoulli | Binary Cross-Entropy |
| Multi-class | Categorical | Softmax + Cross-Entropy |


In [ ]:
# Comparative summary: behavior of different losses on the same data

errors = np.linspace(-3, 3, 500)

losses = {
    "MSE (L2)": errors**2,
    "MAE (L1)": np.abs(errors),
    "Huber (d=1)": np.where(np.abs(errors)<1, 0.5*errors**2, np.abs(errors)-0.5),
    "Log-Cosh": np.log(np.cosh(errors)),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors_list = ['steelblue', 'crimson', 'darkorange', 'purple']
for (name, loss), color in zip(losses.items(), colors_list):
    ax.plot(errors, loss, label=name, lw=2, color=color)

ax.set_title("Loss Function Comparison", fontsize=14)
ax.set_xlabel("Residual (y - y_hat)"); ax.set_ylabel("Loss")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.set_ylim(0, 6)
plt.tight_layout()
plt.savefig("fig_nb03_all_losses.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Summary

- **MLE principle**: Find parameters that best explain the data → minimize NLL
- **Regression** (Gaussian): MSE loss — sensitive to large errors
- **Binary classification** (Bernoulli): BCE — sigmoid + log loss
- **Multi-class** (Categorical): Softmax + Cross-Entropy
- Loss choice depends on the **distributional assumption**

